In [1]:
from __future__ import annotations
import numpy as np
import pandas as pd
from scipy.stats import nbinom, norm

from deconveil.utils_fit import *
from deconveil.utils_processing import *

import rpy2.robjects as ro
from rpy2.robjects import pandas2ri, conversion, Formula
import rpy2.robjects.packages as rpackages
from rpy2.robjects.packages import importr

import matplotlib.pyplot as plt
import seaborn as sns

import pickle
import os
import re
from itertools import product
from glob import glob

In [3]:
import pydeseq2
from pydeseq2.dds import DeseqDataSet
from pydeseq2.default_inference import DefaultInference
from pydeseq2.ds import DeseqStats

from deconveil.dds import deconveil_fit
from deconveil.inference import Inference
from deconveil.default_inference import DefInference
from deconveil.utils_fit import *
from deconveil.utils_processing import *
from deconveil.utils_plot import *
from deconveil import deconveil_fit
from deconveil.ds import deconveil_stats
from deconveil.simulate_gene_dosage import *

#### Load data

In [5]:
rna_counts = load_test_data(
    modality="rna",
    dataset="tcga_brca",
    debug=False,
)
rna_counts = rna_counts.T

metadata = load_test_data(
    modality="metadata",
    dataset="tcga_brca",
    debug=False,
)

cnv = load_test_data(
    modality="cnv",
    dataset="tcga_brca",
    debug=False,
)
cnv = cnv.T

#### QC & filtering

In [7]:
print("Before filtering:", rna_counts.shape, cnv.shape, metadata.shape)
# Reorder cnv to match rna_counts
cnv = cnv.loc[rna_counts.index]
assert (rna_counts.index == metadata.index).all(), "Sample order mismatch between rna_counts and metadata"
assert (cnv.index == rna_counts.index).all(), "Sample order mismatch between cnv and rna_counts"

Before filtering: (400, 17387) (400, 17387) (400, 1)


In [9]:
all_zero_mask = (rna_counts.sum(axis=0) == 0)
print("All-zero genes:", all_zero_mask.sum())
rna_counts = rna_counts.loc[:, ~all_zero_mask]
cnv = cnv.loc[:, ~all_zero_mask]  

All-zero genes: 0


In [11]:
res = filter_low_count_genes(rna_counts, other_dfs=[cnv], min_count=300, min_samples=50)
rna_counts = res["filtered_df"]
cnv = res["other_filtered"][0]
print("After low-count filtering:", rna_counts.shape, cnv.shape)

After low-count filtering: (400, 16252) (400, 16252)


#### Configuration setup

In [43]:
out_dir = "sim_results/sim_data/sim2/"
os.makedirs(out_dir, exist_ok=True)


n_genes = 5000
sample_sizes = [10, 20, 40, 60]

CN_SIGNAL_REGIMES = {
    "CNweak": {
        "dsg_strength": (1.2, 1.6),
        "dcg_strength": (1.2, 2.0),
    },
    "CNstrong": {
        "dsg_strength": (2.0, 3.4),
        "dcg_strength": (2.0, 3.2),
    },
}

# number of replicates per configuration
n_reps = 20
base_seed = 13

In [13]:
cnv = cnv.T
cn_tumor = cnv.iloc[:, 200:400]
cn_tumor = cn_tumor.clip(lower=0, upper=6)
cn_tumor.head()

,TCGA-D8-A1Y1-01A,TCGA-AO-A124-01A,TCGA-AO-A1KS-01A,TCGA-EW-A6SB-01A,TCGA-A8-A08J-01A,TCGA-BH-A0GY-01A,TCGA-AC-A2QJ-01A,TCGA-B6-A401-01A,TCGA-Z7-A8R5-01A,TCGA-A8-A093-01A,...,TCGA-BH-A208-01A,TCGA-A2-A0CL-01A,TCGA-LL-A6FR-01A,TCGA-E9-A1N6-01A,TCGA-AR-A251-01A,TCGA-A2-A4S1-01A,TCGA-E2-A2P5-01A,TCGA-A2-A0EV-01A,TCGA-LL-A6FQ-01A,TCGA-A2-A04T-01A
A2M,3,5,4,6,4,4,2,3,3,3,...,3,4,2,3,2,2,4,2,4,6
A2M-AS1,3,5,4,6,4,4,2,3,3,3,...,3,4,2,3,2,2,4,2,4,6
A2ML1,3,4,4,6,4,4,2,3,3,3,...,3,4,2,5,2,2,4,2,4,6
A2ML1-AS1,3,4,4,6,4,4,2,3,3,3,...,3,4,2,5,2,2,4,2,4,6
A2MP1,3,5,4,6,4,4,2,3,3,3,...,3,4,2,3,2,2,4,2,4,6


In [15]:
cn_normal = cnv.iloc[:, 0:200]
cn_normal = cn_normal.clip(lower=2, upper=2)
cn_normal.head()

,GTEX-13PVQ-1026-SM-5KM3M.1,GTEX-18QFQ-0826-SM-718AX.1,GTEX-1JN6P-2426-SM-ARL99.1,GTEX-13S86-1226-SM-5S2OA.1,GTEX-132NY-0826-SM-5K7Y7.1,GTEX-ZTX8-1226-SM-4YCE9.1,GTEX-UJHI-1426-SM-3DB9C.1,GTEX-15RJE-2626-SM-7KFT1.1,GTEX-ZEX8-2226-SM-57WC6.1,GTEX-1KANA-2026-SM-DIPFB.1,...,GTEX-17F96-2426-SM-7IGLN.1,GTEX-S7SE-0826-SM-4AT4D.1,GTEX-1IDJF-2326-SM-ARL86.1,GTEX-ZF3C-2326-SM-5S2MZ.1,GTEX-1J8EW-2626-SM-CL53E.1,GTEX-R3RS-0626-SM-48FE1.1,GTEX-13O61-1826-SM-5KM4I.1,GTEX-1J8QM-2026-SM-ARZMW.1,GTEX-183FY-1126-SM-7DHLJ.1,GTEX-1B98T-0726-SM-7939J.1
A2M,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
A2M-AS1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
A2ML1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
A2ML1-AS1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2
A2MP1,2,2,2,2,2,2,2,2,2,2,...,2,2,2,2,2,2,2,2,2,2


In [17]:
rna_counts = rna_counts.T
rna_counts.head()

,GTEX-13PVQ-1026-SM-5KM3M.1,GTEX-18QFQ-0826-SM-718AX.1,GTEX-1JN6P-2426-SM-ARL99.1,GTEX-13S86-1226-SM-5S2OA.1,GTEX-132NY-0826-SM-5K7Y7.1,GTEX-ZTX8-1226-SM-4YCE9.1,GTEX-UJHI-1426-SM-3DB9C.1,GTEX-15RJE-2626-SM-7KFT1.1,GTEX-ZEX8-2226-SM-57WC6.1,GTEX-1KANA-2026-SM-DIPFB.1,...,TCGA-BH-A208-01A,TCGA-A2-A0CL-01A,TCGA-LL-A6FR-01A,TCGA-E9-A1N6-01A,TCGA-AR-A251-01A,TCGA-A2-A4S1-01A,TCGA-E2-A2P5-01A,TCGA-A2-A0EV-01A,TCGA-LL-A6FQ-01A,TCGA-A2-A04T-01A
A2M,3502210.0,5394960.0,7136304.0,5710693.0,4020951.0,4662838.0,2417958.0,11106464.0,2393132.0,5335893.0,...,11194647.0,6703305.0,4730868.0,1644905.0,5994136.0,1402228.0,2872243.0,2919369.0,1323810.0,3827589.0
A2M-AS1,83739.0,155914.0,219925.0,236061.0,155161.0,135397.0,62922.0,358297.0,107984.0,130254.0,...,195693.0,115080.0,680762.0,40599.0,104901.0,99105.0,79866.0,58999.0,119745.0,123720.0
A2ML1,1592.0,14511.0,2254.0,932.0,6312.0,2020.0,1644.0,2209.0,1404.0,2513.0,...,558914.0,13736.0,37602.0,9311.0,1361008.0,111.0,2167.0,681.0,198.0,4399482.0
A2ML1-AS1,17369.0,10089.0,12267.0,4446.0,5744.0,4094.0,2319.0,28098.0,8396.0,7131.0,...,2120.0,1285.0,326.0,2331.0,6109.0,351.0,1166.0,1631.0,293.0,3528.0
A2MP1,9842.0,11023.0,5955.0,12644.0,5015.0,7609.0,2149.0,45583.0,7827.0,10049.0,...,5414.0,8559.0,818.0,902.0,2792.0,755.0,3489.0,1801.0,426.0,4632.0


#### Run simulation set up

In [51]:
for S in sample_sizes:
    for cn_label, cn_cfg in CN_SIGNAL_REGIMES.items():
        for R in range(1, n_reps + 1):

            seed = base_seed + R

            print(
                f"Running S={S}, G={n_genes}, "
                f"CN={cn_label}, R={R}"
            )

            sim = cn_aware_rna_simulator(
                counts=rna_counts,
                metadata=metadata,
                cn_tumor=cn_tumor,
                cn_normal=cn_normal,
                n_genes=n_genes,
                design="~ 1",
                seed=seed,
                inject_del=True,
                del_frac_dcg=0.15,
                del_frac_dsg=0.2,
                frac_cn0=0.3,
                max_log2fc=3.0,
                n_normal_sim=S,
                n_tumor_sim=S,
                bootstrap_cn=True,
                # CN signal regime
                dsg_strength=cn_cfg["dsg_strength"],
                dcg_strength=cn_cfg["dcg_strength"],
                cn_heterogeneity_sd=0.40,
                disp_scale_norm=1.0, 
                disp_scale_tum=1.5,
                diff_disp_range=(0.5, 1.5),
                
            )

            fname = (
                f"sim_S{S}_G{n_genes}_"
                f"{cn_label}_R{R}.pkl"
            )

            with open(os.path.join(out_dir, fname), "wb") as f:
                pickle.dump(sim, f)

Running S=10, G=5000, CN=CNweak, R=1


R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 605 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  
R callback write-console: In aggiunta:   
R callback write-console: Messaggio di avvertimento:
  
R callback write-console: In (function (object, test = c("Wald", "LRT"), fitType = c("parametric",  :  
R callback write-console: 
   
R callback write-console:  the design is ~ 1 (just an intercept). is this intended?
  


[Simulator] Using 4991 genes.
Running S=10, G=5000, CN=CNweak, R=2


R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 549 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  
R callback write-console: In aggiunta:   
R callback write-console: Messaggio di avvertimento:
  
R callback write-console: In (function (object, test = c("Wald", "LRT"), fitType = c("parametric",  :  
R callback write-console: 
   
R callback write-console:  the design is ~ 1 (just an intercept). is this intended?
  


[Simulator] Using 4991 genes.
Running S=10, G=5000, CN=CNweak, R=3


...

[Simulator] Using 4991 genes.
Running S=60, G=5000, CN=CNstrong, R=19


R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 584 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  
R callback write-console: In aggiunta:   
R callback write-console: Messaggio di avvertimento:
  
R callback write-console: In (function (object, test = c("Wald", "LRT"), fitType = c("parametric",  :  
R callback write-console: 
   
R callback write-console:  the design is ~ 1 (just an intercept). is this intended?
  


[Simulator] Using 4993 genes.
Running S=60, G=5000, CN=CNstrong, R=20


R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 600 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  
R callback write-console: In aggiunta:   
R callback write-console: Messaggio di avvertimento:
  
R callback write-console: In (function (object, test = c("Wald", "LRT"), fitType = c("parametric",  :  
R callback write-console: 
   
R callback write-console:  the design is ~ 1 (just an intercept). is this intended?
  


[Simulator] Using 4989 genes.


#### Load and extract simulation data

In [53]:
def load_sim(path):
    with open(path, "rb") as f:
        sim = pickle.load(f)
    return sim

def extract_sim_components(sim):

    counts_df = sim["counts"]        # NB-sampled counts (G × N)
    cn_df     = sim["CN"]            # integer CN matrix (G × N)
    truth_df  = sim["truth"]         # truth LFC + class labels
    meta      = sim["metadata"].copy()  # sample annotation

    # Ensure condition labels match A/B format
    meta["condition"] = meta["condition"].replace({
        "normal": "A",
        "tumor": "B"
    })

    return counts_df, cn_df, truth_df, meta

#### Run PyDESeq2 and DeConveil

Helper functions

In [19]:
def run_pydeseq2(rna_counts, metadata, design="~condition", alpha=0.05):
    """
    Runs PyDESeq2 analysis and saves the results.

    Parameters:
        rna_counts (pd.DataFrame): Count matrix with genes as rows and samples as columns.
        metadata (pd.DataFrame): Metadata for the samples with design factors.
        output_path (str): Directory to save the results.
        design_factors (str): Column in metadata to use for design.
        alpha (float): Significance level for statistical tests.
    """
    #os.makedirs(output_path, exist_ok=True)
    
    # Initialize DESeq2 analysis
    inference = DefaultInference(n_cpus=8)
    dds = DeseqDataSet(
        counts=rna_counts,
        metadata=metadata,
        design_factors=None,  # compare samples based on condition
        refit_cooks=True,
        inference=inference,
    )

    # Fit DESeq2 model
    dds.fit_size_factors()
    dds.fit_genewise_dispersions()
    dds.fit_dispersion_trend()
    dds.fit_dispersion_prior()
    dds.fit_MAP_dispersions()
    dds.fit_LFC()
    dds.calculate_cooks()
    
    if dds.refit_cooks:
        dds.refit()

    # Perform statistical analysis
    stat_res_pydeseq = DeseqStats(dds, 
                                  alpha=alpha, 
                                  cooks_filter=True, 
                                  independent_filter=True, 
                                  contrast=["condition", "B", "A"])
    stat_res_pydeseq.run_wald_test()

    if stat_res_pydeseq.cooks_filter:
        stat_res_pydeseq._cooks_filtering()
    stat_res_pydeseq.p_values

    if stat_res_pydeseq.independent_filter:
        stat_res_pydeseq._independent_filtering()
    else:
        stat_res_pydeseq._p_value_adjustment()

    # Log-fold change shrinkage
    stat_res_pydeseq.lfc_shrink(coeff="condition[T.B]")
    stat_res_pydeseq.summary()

    # Save results
    #results_path = os.path.join(output_path, "res_CNnaive.csv")
    #stat_res_pydeseq.results_df.to_csv(results_path)
    return(stat_res_pydeseq.results_df)


def run_deconveil(rna_counts, metadata, cnv, design="~condition", alpha=0.05):
    """
    Runs DeConveil analysis and saves the results.

    Parameters:
        rna_counts (pd.DataFrame): Count matrix with genes as rows and samples as columns.
        metadata (pd.DataFrame): Metadata for the samples with design factors.
        cnv (pd.DataFrame): Copy number variation (CNV) data matrix  with genes as rows and samples as columns.
        output_path (str): Directory to save the results.
        design_factors (str): Column in metadata to use for design.
        alpha (float): Significance level for statistical tests.
    """
    #os.makedirs(output_path, exist_ok=True)
    
    # Initialize DeConveil inference
    inference = DefInference(n_cpus=8)

    # Fit DeConveil model
    dds = deconveil_fit(
        counts=rna_counts,
        metadata=metadata,
        cnv=cnv,
        design_factors="condition",
        inference=inference,
        refit_cooks=True
    )
    dds.fit_size_factors()
    dds.fit_genewise_dispersions()
    dds.fit_dispersion_trend()
    dds.fit_dispersion_prior()
    dds.fit_MAP_dispersions()
    dds.fit_LFC()
    dds.calculate_cooks()

    if dds.refit_cooks:
        dds.refit()  # Replace outlier counts

    # Statistical analysis
    stat_res_deconveil = deconveil_stats(
        dds, 
        alpha=alpha, 
        contrast=["condition", "B", "A"],
        independent_filter=True, 
        cooks_filter=True
    )
    stat_res_deconveil.run_wald_test()

    if stat_res_deconveil.independent_filter:
        stat_res_deconveil._independent_filtering()
    else:
        stat_res_deconveil._p_value_adjustment()

    # Log-fold change shrinkage
    stat_res_deconveil.lfc_shrink(coeff="condition[T.B]")
    stat_res_deconveil.summary()

    # Save results
    #results_path = os.path.join(output_path, "res_CNaware.csv")
    #stat_res_deconveil.results_df.to_csv(results_path)
    return(stat_res_deconveil.results_df)

In [57]:
def parse_sim_filename(path):
    """
    Parse sim_S40_G5000_CNstrong_R3.pkl
    """
    fname = os.path.basename(path)

    m = re.match(
        r"sim_S(\d+)_G(\d+)_CN([A-Za-z0-9]+)_R(\d+)\.pkl",
        fname
    )

    if m is None:
        raise ValueError(f"Could not parse filename: {fname}")

    S  = int(m.group(1))
    G  = int(m.group(2))
    CN = m.group(3)
    R  = int(m.group(4))

    return S, G, CN, R


def analyse_one_sim(path, outdir="sim_results/sim_res_fit/sim2"):

    os.makedirs(outdir, exist_ok=True)
    S, G, CN, R = parse_sim_filename(path)

    sim = load_sim(path)
    counts_df, cn_df, truth_df, meta = extract_sim_components(sim)

    counts_df = counts_df.T
    cn_df = cn_df.T

    print(f"[INFO] Running analysis for S={S}, G={G}, CN={CN}, R={R}")

    print("[INFO] Running PyDESeq2 (CN-naive)")
    res_naive = run_pydeseq2(counts_df, meta)

    print("[INFO] Running DeConveil (CN-aware)")
    res_aware = run_deconveil(counts_df, meta, cn_df)

    print("[INFO] Applying stageR")
    _, _, res_naive, res_aware = run_stageR(res_naive, res_aware)

    # File paths
    out_naive = os.path.join(outdir, f"res_CNnaive_S{S}_G{G}_CN{CN}_R{R}.csv")
    out_aware = os.path.join(outdir, f"res_CNaware_S{S}_G{G}_CN{CN}_R{R}.csv")
    out_truth = os.path.join(outdir, f"truth_S{S}_G{G}_CN{CN}_R{R}.csv")

    # Save
    res_naive.to_csv(out_naive, index=True)
    res_aware.to_csv(out_aware, index=True)
    truth_df.to_csv(out_truth, index=True)

    print(f"[SAVED] {out_naive}")
    print(f"[SAVED] {out_aware}")
    print(f"[SAVED] {out_truth}")

    return truth_df, res_naive, res_aware

def analyse_all_sims(
    sim_dir="sim_data",
    outdir="sim_results/sim_res_fit/sim2/",
):

    os.makedirs(outdir, exist_ok=True)

    sim_files = sorted(
        glob(os.path.join(sim_dir, "sim_S*_G*_CN*_R*.pkl"))
    )

    if not sim_files:
        raise RuntimeError("No simulation files found")

    print(f"[INFO] Found {len(sim_files)} simulations")

    for i, path in enumerate(sim_files, 1):
        print("=" * 60)
        print(f"[INFO] ({i}/{len(sim_files)}) {os.path.basename(path)}")

        try:
            analyse_one_sim(path, outdir=outdir)
        except Exception as e:
            print(f"[ERROR] Failed on {path}")
            print(e)

    print("[DONE] All simulations processed")

In [59]:
results = analyse_all_sims(
    sim_dir="sim_results/sim_data/sim2",
    outdir="sim_results/sim_res_fit/sim2/"
)

[INFO] Found 160 simulations
[INFO] (1/160) sim_S10_G5000_CNstrong_R1.pkl
[INFO] Running analysis for S=10, G=5000, CN=strong, R=1
[INFO] Running PyDESeq2 (CN-naive)
Using None as control genes, passed at DeseqDataSet initialization


/opt/anaconda3/lib/python3.11/site-packages/anndata/_core/anndata.py:1792: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 0.93 seconds.

Fitting dispersion trend curve...
... done in 0.08 seconds.

Fitting MAP dispersions...
... done in 1.25 seconds.

Fitting LFCs...
... done in 0.71 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

/opt/anaconda3/lib/python3.11/site-packages/anndata/_core/anndata.py:1792: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
Replacing 323 outlier genes.

/opt/anaconda3/lib/python3.11/site-packages/anndata/_core/anndata.py:1792: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")
/opt/anaconda3/lib/p

Log2 fold change & Wald test p-value: condition B vs A
                baseMean  log2FoldChange     lfcSE      stat        pvalue  \
JMJD6       3.026211e+05        0.771798  0.256397  3.144974  1.661016e-03   
PLA2G12AP1  4.142646e+03       -0.114618  0.332653 -0.397677  6.908685e-01   
VEGFA       3.434603e+06        3.433408  1.042302  3.503401  4.593572e-04   
RHO         2.638783e+02        0.504632  1.128042 -0.326236  7.442458e-01   
TK1         2.143792e+05       -0.657534  0.915289 -1.541141  1.232826e-01   
...                  ...             ...       ...       ...           ...   
SCN9A       1.299745e+04       -0.155733  0.474445 -0.439491  6.603057e-01   
BCL2L2      5.901878e+05        0.769813  1.025092  1.164804  2.440985e-01   
NADSYN1     5.951822e+05        0.575505  0.332611  2.016585  4.373888e-02   
CUL7        1.118385e+06        2.612650  0.333618  8.077930  6.587523e-16   
POGK        4.104011e+05       -0.314046  0.625979 -0.788268  4.305400e-01   

        

/var/folders/mh/vvy722dj18x7xj41n2qbdzs80000gn/T/ipykernel_8324/637833339.py:81: DeprecationWarning: design_factors are deprecated and will soon be removedPlease consider providing a formulaic formula using the design argument instead
  dds = deconveil_fit(
Fitting size factors...
... done in 0.01 seconds.



Using None as control genes, passed at deconveil_fit initialization


...

replace_mask before filtering: (120, 150)
Number of True values in replace_mask: 243
replacement_counts_trimmed shape: (97, 150)


... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.07 seconds.

Fitting LFCs...
... done in 0.07 seconds.

Running Wald tests...
... done in 0.30 seconds.

Fitting MAP LFCs...
... done in 1.05 seconds.



Log2 fold change & Wald test p-value: condition B vs A
                 baseMean  log2FoldChange     lfcSE       stat        pvalue  \
VN1R81P      1.596829e+03       -1.129091  0.181341  -6.233069  4.573854e-10   
MIR7844      1.992146e+03       -1.991544  0.299405  -7.162657  7.912793e-13   
CFAP69       4.729812e+04       -0.454007  0.294500 -13.305311  2.155954e-40   
BCAR3        1.957117e+05        1.505826  0.190689  -0.270201  7.870055e-01   
CEP57        2.158584e+05        1.596479  0.120739  -0.825569  4.090486e-01   
...                   ...             ...       ...        ...           ...   
ARPC4-TTLL3  1.110465e+06        2.773561  0.082359   9.897944  4.249215e-23   
SRGAP3-AS4   1.144677e+02       -1.224161  0.224027  -5.676310  1.376312e-08   
GPR45        2.351822e+02       -1.117716  0.589004  -2.478390  1.319767e-02   
SULT4A1      8.350744e+03        0.375796  0.325844   1.541145  1.232814e-01   
GTSE1-DT     2.686052e+03        0.107934  0.108259   0.866898  3

R callback write-console: The returned adjusted p-values are based on a stage-wise testing approach and are only valid for the provided target OFDR level of 5%. If a different target OFDR level is of interest,the entire adjustment should be re-run. 

  
R callback write-console: The returned adjusted p-values are based on a stage-wise testing approach and are only valid for the provided target OFDR level of 5%. If a different target OFDR level is of interest,the entire adjustment should be re-run. 

  


[SAVED] sim_results/sim_res_fit/sim2/res_CNnaive_S60_G5000_CNweak_R9.csv
[SAVED] sim_results/sim_res_fit/sim2/res_CNaware_S60_G5000_CNweak_R9.csv
[SAVED] sim_results/sim_res_fit/sim2/truth_S60_G5000_CNweak_R9.csv
[DONE] All simulations processed


#### Null simulation to show type-I error calibration

In [41]:
def collect_null_pvals(
    rna_counts,
    metadata,
    cn_tumor,
    cn_normal,
    n_genes=5000,
    sample_sizes=(10, 20, 40, 60),
    n_reps=20,
    seed_base=2000,
):
    all_p = []

    for S in sample_sizes:
        for rep in range(n_reps):
            seed = seed_base + rep
            sim = cn_aware_rna_simulator(
            counts=rna_counts,
            metadata=metadata,
            cn_tumor=cn_tumor,
            cn_normal=cn_normal,
            n_genes=n_genes,
            design="~ 1",
            n_normal_sim=S,
            n_tumor_sim=S,
            seed=seed,
            null_mode="pure_null",  
            inject_del=False,     # usually disable extra structure in null
            )
            counts_df = sim["counts"].T
            cn_df     = sim["CN"].T
            meta      = sim["metadata"]
            meta["condition"] = meta["condition"].replace({"normal": "A", "tumor": "B"})

            # PyDESeq2 (CN-naive)
            res_naive = run_pydeseq2(counts_df, meta)
            p_naive = res_naive["pvalue"].astype(float).values

            # DeConveil (CN-aware)
            res_aware = run_deconveil(counts_df, meta, cn_df)
            p_aware = res_aware["pvalue"].astype(float).values

            all_p.append(pd.DataFrame({
                "p_naive": p_naive,
                "p_aware": p_aware,
                "sample_size": S,
                "rep": rep,
            }))

    return pd.concat(all_p, ignore_index=True)

In [43]:
all_p = collect_null_pvals(rna_counts, metadata, cn_tumor, cn_normal)

R callback write-console: estimating size factors
  
R callback write-console: estimating dispersions
  
R callback write-console: gene-wise dispersion estimates
  
R callback write-console: mean-dispersion relationship
  
R callback write-console: final dispersion estimates
  
R callback write-console: fitting model and testing
  
R callback write-console: -- replacing outliers and refitting for 577 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)
  
R callback write-console: estimating dispersions
  
R callback write-console: fitting model and testing
  
R callback write-console: In aggiunta:   
R callback write-console: Messaggio di avvertimento:
  
R callback write-console: In (function (object, test = c("Wald", "LRT"), fitType = c("parametric",  :  
R callback write-console: 
   
R callback write-console:  the design is ~ 1 (just an intercept). is this intended?
  


[Simulator] Using 4994 genes.
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.00 seconds.

Fitting dispersions...
... done in 1.09 seconds.

Fitting dispersion trend curve...
... done in 0.12 seconds.

Fitting MAP dispersions...
... done in 1.39 seconds.

Fitting LFCs...
... done in 1.21 seconds.

Calculating cook's distance...
... done in 0.02 seconds.

Replacing 267 outlier genes.

Fitting dispersions...
... done in 0.07 seconds.

Fitting MAP dispersions...
... done in 0.08 seconds.

Fitting LFCs...
... done in 0.10 seconds.

Running Wald tests...
... done in 0.24 seconds.

Fitting MAP LFCs...
/opt/anaconda3/lib/python3.11/site-packages/pydeseq2/utils.py:1088: RuntimeWarning: overflow encountered in exp
  counts - (counts + size) / (1 + size * np.exp(-xbeta - offset))
/opt/anaconda3/lib/python3.11/site-packages/pydeseq2/utils.py:1088: RuntimeWarning: overflow encountered in exp
  counts - (counts + size) / (1 + size * np.exp(-xbeta - offset))
... done in 1.53 seconds.



Log2 fold change & Wald test p-value: condition B vs A
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
RTKN       227682.004884        0.025787  0.120375  1.835457  0.066438   
RPL31P2      2854.599577        0.003642  0.100382  0.468735  0.639259   
MIR6746      1107.543069       -0.002397  0.088767 -0.529412  0.596520   
CYTH2      577205.361977        0.016486  0.102059  1.118404  0.263395   
RNF112      25607.049404       -0.001581  0.092849 -0.260533  0.794453   
...                  ...             ...       ...       ...       ...   
MYL1         1454.668783        0.001077  0.102238  0.269102  0.787851   
PPIAP16       385.760544        0.016385  0.114559  1.698999  0.089319   
LINC02084    1364.721882        0.002065  0.091499  0.123027  0.902086   
MIR202HG      533.817869       -0.031959  0.098524 -1.774575  0.075968   
LRIG3      165343.742422        0.001999  0.099760  0.304053  0.761088   

               padj  
RTKN       0.729078  
RPL31P2    0

/var/folders/mh/vvy722dj18x7xj41n2qbdzs80000gn/T/ipykernel_3408/637833339.py:81: DeprecationWarning: design_factors are deprecated and will soon be removedPlease consider providing a formulaic formula using the design argument instead
  dds = deconveil_fit(
Fitting size factors...
... done in 0.01 seconds.



...

Fitting dispersions...
... done in 0.95 seconds.

Fitting dispersion trend curve...
... done in 0.08 seconds.

Fitting MAP dispersions...
... done in 1.42 seconds.

Fitting LFCs...
... done in 0.79 seconds.

Calculating cook's distance...
... done in 0.06 seconds.

Replacing 114 outlier genes.

Fitting dispersions...
... done in 0.05 seconds.

Fitting MAP dispersions...


replace_mask before filtering: (120, 114)
Number of True values in replace_mask: 186
replacement_counts_trimmed shape: (95, 114)


... done in 0.05 seconds.

Fitting LFCs...
... done in 0.05 seconds.

Running Wald tests...
... done in 0.30 seconds.

Fitting MAP LFCs...
/Users/katsiarynadavydzenka/Documents/PhD_AI/DeConveil/deconveil/utils_fit.py:422: RuntimeWarning: overflow encountered in exp
  counts - (counts + size) / (1 + size * np.exp(-xbeta - offset - cnv))


Log2 fold change & Wald test p-value: condition B vs A
                baseMean  log2FoldChange     lfcSE      stat    pvalue  \
HMMR-AS1     2158.345037        0.000054  0.011703  0.329049  0.742119   
ZNRF2P2      3509.094133        0.000314  0.011803  0.888229  0.374418   
CCT6B       21917.614866        0.354819  0.211203  0.170867  0.864328   
CCDC201       266.154361        0.000044  0.012526  0.580018  0.561903   
HDGFL1         21.300028        0.000260  0.012806 -3.347050  0.000817   
...                  ...             ...       ...       ...       ...   
CCDC162P    20119.137205        0.000113  0.011412  0.135845  0.891944   
LINC00240    7115.401971        0.000058  0.010684 -0.774217  0.438802   
DDX27      343287.251007        2.005680  0.249857  1.220883  0.222130   
DEGS1      556872.472264        1.621036  0.123905 -1.123782  0.261105   
RPS3AP37      142.746428        0.000308  0.011777  0.946636  0.343824   

               padj  
HMMR-AS1   0.989756  
ZNRF2P2    0

... done in 1.28 seconds.



In [45]:
all_p.to_csv("sim_results/sim_null/null_pval_all_v3.csv", index=True)